In [2]:
import numpy as np
import pandas as pd
import pywt, cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

data = pd.read_csv("/kaggle/input/beed-dataset/BEED_Data.csv")

print("Dataset shape:", data.shape)
print(data.head())

X_all = data.iloc[:, :-1].values
y_all = data.iloc[:, -1].values

print("EEG shape:", X_all.shape)
print("Labels shape:", y_all.shape)

def eeg_to_cwt(eeg_signal, scales=np.arange(1, 65), wavelet='morl', size=128):
    coef, _ = pywt.cwt(eeg_signal, scales, wavelet)
    img = np.abs(coef)
    img = cv2.resize(img, (size, size))
    return img

X_images = np.array([eeg_to_cwt(sig) for sig in X_all])
X_images = X_images[..., np.newaxis]  

print("CWT image dataset shape:", X_images.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_images, y_all, test_size=0.2, stratify=y_all, random_state=42
)

num_classes = len(np.unique(y_all))
y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,1)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    verbose=1
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"BEED Test Accuracy: {acc:.2f}")


Dataset shape: (8000, 17)
    X1   X2   X3   X4   X5   X6  X7  X8  X9  X10  X11  X12  X13  X14  X15  \
0    4    7   18   25   28   27  20  10 -10  -18  -20  -16   13   32   12   
1   87  114  120  106   76   54  28   5 -19  -49  -85 -102 -100  -89  -61   
2 -131 -133 -140 -131 -123 -108 -58 -51 -70  -77  -76  -76  -73  -57  -40   
3   68  104   73   34  -12  -26 -38 -36 -67  -88  -25   31   18   -4    6   
4  -67  -90  -97  -94  -86  -71 -43 -11  23   46   58   50   39   19   -9   

   X16  y  
0   10  0  
1  -21  0  
2  -14  0  
3  -29  0  
4  -41  0  
EEG shape: (8000, 16)
Labels shape: (8000,)
CWT image dataset shape: (8000, 128, 128, 1)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,476 (12.61 MB)

 Trainable params: 3,305,028 (12.61 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.5571 - loss: 1.6935 - val_accuracy: 0.4859 - val_loss: 1.4734
Epoch 2/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6372 - loss: 0.7635 - val_accuracy: 0.6469 - val_loss: 0.7139
Epoch 3/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6203 - loss: 0.7555 - val_accuracy: 0.5359 - val_loss: 0.8443
Epoch 4/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6646 - loss: 0.6928 - val_accuracy: 0.6680 - val_loss: 0.6749
Epoch 5/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.7168 - loss: 0.6062 - val_accuracy: 0.7961 - val_loss: 0.4859
Epoch 6/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.7527 - loss: 0.5795 - val_accuracy: 0.7812 - val_loss: 0.6359
Epoch 7/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.7659 - loss: 0.5278 - val_accuracy: 0.8055 - val_loss: 0.5219
Epoch 8/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.7822 - loss: 0.4832 - val_acc

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
data = pd.read_csv("/kaggle/input/beed-dataset/BEED_Data.csv")

X_all = data.iloc[:, :-1].values  # EEG signals
y_all = data.iloc[:, -1].values   # Labels

print("EEG shape:", X_all.shape)
print("Labels shape:", y_all.shape)

# Function to extract FFT features
def eeg_to_fft_features(eeg_signal):
    fft_vals = np.fft.fft(eeg_signal)
    fft_abs = np.abs(fft_vals)[:len(fft_vals)//2]  # Take only positive frequencies
    fft_norm = fft_abs / np.sum(fft_abs)           # Normalize
    return fft_norm

# Transform EEG dataset to FFT features
X_fft = np.array([eeg_to_fft_features(sig) for sig in X_all])
print("FFT feature dataset shape:", X_fft.shape)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_fft, y_all, test_size=0.2, stratify=y_all, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# --- SVM Classifier ---
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"SVM Test Accuracy: {acc_svm:.2f}")
print("SVM Classification Report:\n", classification_report(y_test, y_pred_svm))

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Test Accuracy: {acc_rf:.2f}")
print("Random Forest Classification Report:\n", classification_report(y_test, y_pred_rf))


EEG shape: (8000, 16)
Labels shape: (8000,)
FFT feature dataset shape: (8000, 8)
SVM Test Accuracy: 0.66
SVM Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       400
           1       0.57      0.54      0.55       400
           2       0.53      0.57      0.55       400
           3       0.57      0.55      0.56       400

    accuracy                           0.66      1600
   macro avg       0.66      0.66      0.66      1600
weighted avg       0.66      0.66      0.66      1600

Random Forest Test Accuracy: 0.75
Random Forest Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      1.00       400
           1       0.70      0.70      0.70       400
           2       0.64      0.63      0.64       400
           3       0.67      0.69      0.68       400

    accuracy                           0.75      1600
   macro avg       0.75      0.75 

In [2]:
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier

voting_model = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)),
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'))
    ],
    voting='soft'
)

voting_model.fit(X_train, y_train)
y_pred = voting_model.predict(X_test)
print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred))


Voting Classifier Accuracy: 0.713125


In [ ]:
# hyper Tuning

In [4]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Load dataset
# -----------------------------
data = pd.read_csv("/kaggle/input/beed-dataset/BEED_Data.csv")
X_all = data.iloc[:, :-1].values  # EEG signals
y_all = data.iloc[:, -1].values   # Labels

print("EEG shape:", X_all.shape)
print("Labels shape:", y_all.shape)

# -----------------------------
# Feature extraction
# -----------------------------
def eeg_features(eeg_signal, fft_points=128):
    """
    Extract time-domain and frequency-domain features from a single EEG signal.
    """
    # Time-domain stats
    time_feats = [
        np.mean(eeg_signal),
        np.std(eeg_signal),
        np.var(eeg_signal),
        skew(eeg_signal),
        kurtosis(eeg_signal)
    ]
    
    # Frequency-domain features (FFT with zero-padding)
    fft_vals = np.fft.fft(eeg_signal, n=fft_points)
    fft_abs = np.abs(fft_vals)[:fft_points // 2]  # take positive frequencies
    fft_norm = fft_abs / np.sum(fft_abs)          # normalize
    
    return np.concatenate([time_feats, fft_norm])

# Apply feature extraction to all EEG signals
X_features = np.array([eeg_features(sig) for sig in X_all])
print("Feature dataset shape:", X_features.shape)

# -----------------------------
# Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_all, test_size=0.2, stratify=y_all, random_state=42
)

# -----------------------------
# Feature scaling
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# PCA for dimensionality reduction
# -----------------------------
# Choose n_components safely
n_components = min(50, X_train.shape[1])
pca = PCA(n_components=n_components)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print("PCA-transformed feature shape:", X_train.shape)

# -----------------------------
# Ensemble Model: Random Forest + XGBoost
# -----------------------------
rf_model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42)
xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

voting_model = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting='soft'
)

# Train the ensemble
voting_model.fit(X_train, y_train)

# -----------------------------
# Evaluate
# -----------------------------
y_pred = voting_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Improved Ensemble Test Accuracy: {acc:.2f}")
print("Classification Report:\n", classification_report(y_test, y_pred))


EEG shape: (8000, 16)
Labels shape: (8000,)
Feature dataset shape: (8000, 69)
PCA-transformed feature shape: (6400, 50)
Improved Ensemble Test Accuracy: 0.83
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       0.83      0.78      0.80       400
           2       0.75      0.81      0.78       400
           3       0.75      0.75      0.75       400

    accuracy                           0.83      1600
   macro avg       0.83      0.83      0.83      1600
weighted avg       0.83      0.83      0.83      1600



In [ ]:
# Further Improvement

In [5]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.signal import welch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Load dataset
# -----------------------------
data = pd.read_csv("/kaggle/input/beed-dataset/BEED_Data.csv")
X_all = data.iloc[:, :-1].values  # EEG signals
y_all = data.iloc[:, -1].values   # Labels

print("EEG shape:", X_all.shape)
print("Labels shape:", y_all.shape)

# -----------------------------
# Feature extraction
# -----------------------------
def eeg_band_powers(eeg_signal, fs=128):
    """
    Compute EEG band powers using Welch's method.
    Bands: Delta(0.5-4Hz), Theta(4-8Hz), Alpha(8-13Hz), Beta(13-30Hz), Gamma(30-50Hz)
    """
    f, Pxx = welch(eeg_signal, fs=fs, nperseg=min(len(eeg_signal), 256))
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 50)
    }
    band_powers = []
    for band in bands.values():
        idx = np.logical_and(f >= band[0], f <= band[1])
        power = np.sum(Pxx[idx])
        band_powers.append(power)
    return np.array(band_powers)

def eeg_features_optimized(eeg_signal, fft_points=128, fs=128):
    # Time-domain stats
    time_feats = [
        np.mean(eeg_signal),
        np.std(eeg_signal),
        np.var(eeg_signal),
        skew(eeg_signal),
        kurtosis(eeg_signal)
    ]
    
    # Frequency-domain FFT features
    fft_vals = np.fft.fft(eeg_signal, n=fft_points)
    fft_abs = np.abs(fft_vals)[:fft_points // 2]
    fft_norm = fft_abs / np.sum(fft_abs)
    
    # EEG band powers
    band_feats = eeg_band_powers(eeg_signal, fs)
    
    # Combine all features
    return np.concatenate([time_feats, fft_norm, band_feats])

# Extract features for all EEG signals
X_features = np.array([eeg_features_optimized(sig) for sig in X_all])
print("Optimized feature dataset shape:", X_features.shape)

# -----------------------------
# Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_all, test_size=0.2, stratify=y_all, random_state=42
)

# -----------------------------
# Feature scaling
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# PCA for dimensionality reduction
# -----------------------------
n_components = min(50, X_train.shape[1])
pca = PCA(n_components=n_components)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print("PCA-transformed feature shape:", X_train.shape)

# -----------------------------
# Ensemble Model: Random Forest + XGBoost
# -----------------------------
rf_model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42)
xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

voting_model = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting='soft'
)

# Train ensemble
voting_model.fit(X_train, y_train)

# -----------------------------
# Evaluate
# -----------------------------
y_pred = voting_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Optimized Ensemble Test Accuracy: {acc:.2f}")
print("Classification Report:\n", classification_report(y_test, y_pred))


EEG shape: (8000, 16)
Labels shape: (8000,)
Optimized feature dataset shape: (8000, 74)
PCA-transformed feature shape: (6400, 50)
Optimized Ensemble Test Accuracy: 0.84
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       0.84      0.81      0.82       400
           2       0.76      0.80      0.78       400
           3       0.76      0.75      0.75       400

    accuracy                           0.84      1600
   macro avg       0.84      0.84      0.84      1600
weighted avg       0.84      0.84      0.84      1600



In [6]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.signal import welch
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Load dataset
# -----------------------------
data = pd.read_csv("/kaggle/input/beed-dataset/BEED_Data.csv")
X_all = data.iloc[:, :-1].values
y_all = data.iloc[:, -1].values
print("EEG shape:", X_all.shape)
print("Labels shape:", y_all.shape)

# -----------------------------
# Feature extraction functions
# -----------------------------
def eeg_band_powers(eeg_signal, fs=128):
    f, Pxx = welch(eeg_signal, fs=fs, nperseg=min(len(eeg_signal), 256))
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 50)
    }
    return [np.sum(Pxx[np.logical_and(f >= b[0], f <= b[1])]) for b in bands.values()]

def eeg_features_optimized(eeg_signal, fft_points=128, fs=128):
    # Time-domain stats
    time_feats = [
        np.mean(eeg_signal),
        np.std(eeg_signal),
        np.var(eeg_signal),
        skew(eeg_signal),
        kurtosis(eeg_signal)
    ]
    # FFT features
    fft_vals = np.fft.fft(eeg_signal, n=fft_points)
    fft_abs = np.abs(fft_vals)[:fft_points // 2]
    fft_norm = fft_abs / np.sum(fft_abs)
    # Band powers
    band_feats = eeg_band_powers(eeg_signal, fs)
    return np.concatenate([time_feats, fft_norm, band_feats])

# Extract features
X_features = np.array([eeg_features_optimized(sig) for sig in X_all])
print("Feature dataset shape:", X_features.shape)

# -----------------------------
# Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_all, test_size=0.2, stratify=y_all, random_state=42
)

# -----------------------------
# Feature scaling
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# PCA for dimensionality reduction
# -----------------------------
n_components = min(50, X_train.shape[1])
pca = PCA(n_components=n_components)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print("PCA-transformed feature shape:", X_train.shape)

# -----------------------------
# Hyperparameter tuning: Random Forest
# -----------------------------
rf_param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5]
}
rf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf, rf_param_grid, cv=3, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)
print("Best RF params:", rf_grid.best_params_)

# -----------------------------
# Hyperparameter tuning: XGBoost
# -----------------------------
xgb_param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [5, 6, 7],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_grid = GridSearchCV(xgb, xgb_param_grid, cv=3, scoring='accuracy', n_jobs=-1)
xgb_grid.fit(X_train, y_train)
print("Best XGB params:", xgb_grid.best_params_)

# -----------------------------
# Expanded Voting Ensemble
# -----------------------------
extra_tree = ExtraTreesClassifier(n_estimators=300, max_depth=15, random_state=42)

voting_model = VotingClassifier(
    estimators=[
        ('rf', rf_grid.best_estimator_),
        ('xgb', xgb_grid.best_estimator_),
        ('et', extra_tree)
    ],
    voting='soft'
)

# Train ensemble
voting_model.fit(X_train, y_train)

# -----------------------------
# Evaluate
# -----------------------------
y_pred = voting_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Tuned Ensemble Test Accuracy: {acc:.2f}")
print("Classification Report:\n", classification_report(y_test, y_pred))


EEG shape: (8000, 16)
Labels shape: (8000,)
Feature dataset shape: (8000, 74)
PCA-transformed feature shape: (6400, 50)
Best RF params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 300}
Best XGB params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'subsample': 0.8}
Tuned Ensemble Test Accuracy: 0.89
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       0.91      0.86      0.88       400
           2       0.81      0.90      0.85       400
           3       0.85      0.81      0.83       400

    accuracy                           0.89      1600
   macro avg       0.89      0.89      0.89      1600
weighted avg       0.89      0.89      0.89      1600

